# Molecular Quantum Chemistry

This notebook demonstrates the full quantum chemistry workflow:
molecular geometry to QPE, VQE, and resource estimation --
reproducing the Microsoft qdk-chemistry pipeline using PySCF.

**Requirements:** `pip install qdk-pythonic[chemistry]`

## 1. Hartree-Fock State Preparation

The HF state is the starting point for both QPE and VQE.

In [ ]:
from qdk_pythonic.domains.chemistry import HartreeFockState

# H2: 2 electrons in 4 spin-orbitals
hf = HartreeFockState(n_qubits=4, n_electrons=2)
print(f"JW bitstring: {hf.to_bitstring()}")

circuit = hf.to_circuit()
print(f"Circuit: {circuit.qubit_count()} qubits, {circuit.total_gate_count()} gates")
print(circuit.draw())

# Bravyi-Kitaev encoding
hf_bk = HartreeFockState(n_qubits=4, n_electrons=2, mapping="bravyi_kitaev")
print(f"\nBK bitstring: {hf_bk.to_bitstring()}")

## 2. UCCSD Ansatz

The UCCSD ansatz generates parameterized single and double
excitation operators from the HF reference.

In [ ]:
from qdk_pythonic.domains.chemistry import UCCSDAnsatz

ansatz = UCCSDAnsatz(n_spatial_orbitals=2, n_electrons=2)
print(f"Spin-orbitals (qubits): {ansatz.n_qubits}")
print(f"Parameters: {ansatz.num_parameters}")
print(f"Singles: {ansatz.singles()}")
print(f"Doubles: {ansatz.doubles()}")

# Build circuit with concrete parameters
params = [0.1] * ansatz.num_parameters
circuit = ansatz.to_circuit(params)
print(f"\nCircuit: {circuit.qubit_count()} qubits, "
      f"{circuit.total_gate_count()} gates, depth {circuit.depth()}")

## 3. Quantum Phase Estimation (QPE)

QPE uses controlled Hamiltonian simulation and inverse QFT
to extract the ground-state energy.

In [ ]:
from qdk_pythonic.adapters.pyscf_adapter import molecular_hamiltonian
from qdk_pythonic.domains.chemistry import ChemistryQPE

h = molecular_hamiltonian("H 0 0 0; H 0 0 0.74", basis="sto-3g")
h.print_summary()

qpe = ChemistryQPE(
    hamiltonian=h,
    n_electrons=2,
    n_estimation_qubits=4,
    trotter_steps=2,
)
circuit = qpe.to_circuit()
print(f"\nQPE circuit: {circuit.qubit_count()} qubits, "
      f"{circuit.total_gate_count()} gates")
print(f"System qubits: {h.qubit_count()}")
print(f"Estimation qubits: 4")

## 4. VQE Setup

VQE combines a parameterized ansatz with classical optimization.
Here we show the circuit construction (running requires qsharp).

In [ ]:
from qdk_pythonic.domains.chemistry import VQE, UCCSDAnsatz

ansatz = UCCSDAnsatz(n_spatial_orbitals=2, n_electrons=2)
vqe = VQE(
    hamiltonian=h,
    ansatz=ansatz,
    n_electrons=2,
    optimizer="COBYLA",
    max_iterations=100,
)

# Build trial-state circuit
trial = vqe.to_circuit([0.0] * ansatz.num_parameters)
print(f"Trial circuit: {trial.qubit_count()} qubits, "
      f"{trial.total_gate_count()} gates")
print(f"\nTo run VQE optimization (requires qsharp):")
print("  result = vqe.run()")
print("  print(result.optimal_energy)")

## 5. Expectation Value Grouping

VQE measures Pauli terms grouped by qubit-wise commutativity.

In [ ]:
from qdk_pythonic.domains.chemistry import group_commuting_terms

groups = group_commuting_terms(h)
print(f"Hamiltonian has {len(h)} Pauli terms")
print(f"Grouped into {len(groups)} measurement circuits")
for i, g in enumerate(groups):
    print(f"  Group {i}: {len(g)} terms")

## 6. FCIDUMP Interoperability

Read/write molecular integrals in the standard FCIDUMP format.

In [ ]:
from qdk_pythonic.domains.chemistry import FCIDUMPData
from qdk_pythonic.adapters.pyscf_adapter import run_scf, get_integrals
import numpy as np

scf = run_scf("H 0 0 0; H 0 0 0.74", basis="sto-3g")
h1e, h2e, nuc = get_integrals(scf)

data = FCIDUMPData(
    n_orbitals=h1e.shape[0],
    n_electrons=int(scf.mol.nelectron),
    ms2=0,
    h1e=h1e, h2e=h2e,
    nuclear_repulsion=nuc,
)
print(f"{data.n_orbitals} orbitals, {data.n_electrons} electrons")
print(f"h1e shape: {data.h1e.shape}")
print(f"h2e shape: {data.h2e.shape}")

## 7. Orbital Information

In [ ]:
from qdk_pythonic.domains.chemistry import MolecularOrbitalInfo

info = MolecularOrbitalInfo.from_pyscf(scf)
info.print_report()

## 8. Algorithm Registry

All chemistry algorithms are available through the QDK-style registry.

In [ ]:
from qdk_pythonic.registry import create, available

# List registered chemistry algorithms
for algo_type, names in available().items():
    if "chemistry" in algo_type or "pyscf" in str(names):
        print(f"{algo_type}: {names}")